# Evaluation on CROHME

In [ ]:
import torch
import numpy as np
import pickle
import csv
import sys
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from datasets import load_dataset

sys.path.insert(0, "models")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ARTIFACTS = Path("artifacts")

def normalized_edit_distance(s1, s2):
    if len(s1) == 0 and len(s2) == 0: return 0.0
    if len(s1) == 0 or  len(s2) == 0: return 1.0
    d = [[0] * (len(s2) + 1) for _ in range(len(s1) + 1)]
    for i in range(len(s1) + 1): d[i][0] = i
    for j in range(len(s2) + 1): d[0][j] = j
    for i in range(1, len(s1) + 1):
        for j in range(1, len(s2) + 1):
            cost = 0 if s1[i-1] == s2[j-1] else 1
            d[i][j] = min(d[i-1][j]+1, d[i][j-1]+1, d[i-1][j-1]+cost)
    return d[len(s1)][len(s2)] / max(len(s1), len(s2))

def resize_pad_grayscale(img_pil, target=256, pad_value=255):
    img = img_pil.convert("L")
    w, h = img.size
    scale = target / max(w, h)
    new_w, new_h = max(1, int(round(w * scale))), max(1, int(round(h * scale)))
    img_rs = img.resize((new_w, new_h), resample=Image.BICUBIC)
    canvas = Image.new("L", (target, target), color=pad_value)
    canvas.paste(img_rs, ((target - new_w) // 2, (target - new_h) // 2))
    return canvas

# Tokenizer
with open(ARTIFACTS / "lstm_tokenizer92.pkl", "rb") as f:
    tokenizer = pickle.load(f)
VS = max(tokenizer.word_index.values()) + 1
START = VS - 2
END = VS - 1
inv_vocab = {v: k for k, v in tokenizer.word_index.items()}

def decode(seq):
    return "".join(inv_vocab.get(t, "") for t in seq if t not in (0, START, END))

print(f"Device: {DEVICE}, vocab_size={VS}, START={START}, END={END}")

In [ ]:
from vit_lora_lstm_attn import ViTLatexModelLoRA as LSTMModel

model = LSTMModel(vocab_size=VS, lora_r=32, use_rslora=True).to(DEVICE)
ckpt = torch.load(ARTIFACTS / "lstm_rslora_r32.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model"])
model.eval()
print("LSTM RSLoRA r=32 loaded.")

In [ ]:
# ---- Config (edit these) ----
CROHME_SPLITS = ["2014", "2016", "2019"]
CROHME_N = 200        # samples per split, None = use all
CROHME_MIN_LEN = 10   # skip trivially short labels
CROHME_CSV = Path("../data/crohme_eval_results.csv")

In [ ]:
crohme = load_dataset("Neeze/CROHME-full")
for split in crohme:
    print(f"  {split}: {len(crohme[split])} samples")

In [ ]:
results = []

for split in CROHME_SPLITS:
    ds_split = crohme[split]
    all_indices = [i for i in range(len(ds_split)) if len(ds_split[i]["label"]) >= CROHME_MIN_LEN]
    n = CROHME_N if CROHME_N else len(all_indices)
    indices = all_indices[:n]

    print(f"\nCROHME {split}: {len(ds_split)} total, {len(all_indices)} filtered, evaluating {n}")
    print("-" * 60)

    exact = 0
    total_ed = 0.0

    for j, i in enumerate(indices):
        sample = ds_split[i]
        img = resize_pad_grayscale(sample["image"], target=256)
        img = np.array(img, dtype=np.float32) / 255.0
        img_t = torch.tensor(img, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        img_t = img_t.repeat(1, 3, 1, 1).to(DEVICE)

        with torch.no_grad():
            pred_tokens = model.generate_beam(img_t, max_len=150, sos_idx=START, eos_idx=END, beam_size=5)

        pred = decode(pred_tokens).replace(" ", "")
        gt = sample["label"].replace(" ", "")
        ed = normalized_edit_distance(pred, gt)
        if pred == gt:
            exact += 1
        total_ed += ed

        if (j + 1) % 50 == 0 or j == n - 1:
            print(f"  [{j+1:>4}/{n}] exact={exact/(j+1):.2%}, avg ED={total_ed/(j+1):.4f}", flush=True)

    avg_ed = total_ed / n
    results.append({
        "split": split,
        "n": n,
        "exact_match_pct": round(exact / n * 100, 2),
        "avg_edit_distance": round(avg_ed, 4),
    })
    print(f"  FINAL: Exact={exact/n:.2%} ({exact}/{n}), Avg ED={avg_ed:.4f}")

# Save CSV
CROHME_CSV.parent.mkdir(parents=True, exist_ok=True)
with open(CROHME_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["split", "n", "exact_match_pct", "avg_edit_distance"])
    writer.writeheader()
    writer.writerows(results)
print(f"\nSaved to {CROHME_CSV}")

In [ ]:
splits = [r["split"] for r in results]
exact_pcts = [r["exact_match_pct"] for r in results]
avg_eds = [r["avg_edit_distance"] for r in results]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Exact match
axes[0].bar(splits, exact_pcts, color="steelblue")
axes[0].set_ylabel("Exact Match (%)")
axes[0].set_xlabel("CROHME Year")
axes[0].set_title("Exact Match by Year")
for i, v in enumerate(exact_pcts):
    axes[0].text(i, v + 0.5, f"{v:.1f}%", ha="center", fontsize=10)

# Avg edit distance
axes[1].bar(splits, avg_eds, color="coral")
axes[1].set_ylabel("Avg Edit Distance")
axes[1].set_xlabel("CROHME Year")
axes[1].set_title("Avg Edit Distance by Year")
for i, v in enumerate(avg_eds):
    axes[1].text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=10)

fig.suptitle("LSTM RSLoRA r=32 (92-vocab) on CROHME", fontsize=13)
plt.tight_layout()
plt.savefig("../data/crohme_bar_chart.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved chart to data/crohme_bar_chart.png")